# Tarea B — Temperatura, top-p y entropía de la distribución de salida

MMIA 6013 · Tarea de práctica (Taller 03 v2). Vocabulario de seis tokens t0–t5 con logits z = [2.0, 1.0, 0.5, 0.2, −1.0, −3.0]. Todas las cifras de este documento son calculadas por las celdas de código del notebook.

## Parte 1 — Softmax con temperatura

La softmax con temperatura se define como $p_i = \exp(z_i/T) \big/ \sum_j \exp(z_j/T)$. Para implementarla de forma numéricamente estable se resta el máximo de los logits antes de exponenciar, es decir, se calcula $\exp((z_i - z_{\max})/T)$, lo que evita desbordamientos de punto flotante sin alterar el resultado: la verificación contra la softmax estándar (sin desplazamiento) arrojó una diferencia absoluta máxima de 0.0, y cada distribución suma exactamente 1.0.

| Token | Logit | T = 0.5 | T = 1.0 | T = 2.0 |
|---|---:|---:|---:|---:|
| t0 | 2.0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 1.0 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.5 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.2 | 0.0225 | 0.0912 | 0.1457 |
| t4 | −1.0 | 0.0020 | 0.0275 | 0.0800 |
| t5 | −3.0 | 0.0000 | 0.0037 | 0.0294 |
| **Entropía (bits)** | — | **0.9147** | **1.7998** | **2.2888** |

(En T = 0.5 la probabilidad de t5 es 3.7367e-05, que se muestra como 0.0000 al redondear a cuatro decimales.)

La tabla cuantifica el efecto de la temperatura: al bajarla de 2 a 0.5 la masa se concentra en el token de mayor logit (t0 pasa de 0.3583 a 0.8231) y la entropía cae de 2.2888 a 0.9147 bits; al subirla, la distribución se aplana y la entropía crece. La celda de esta parte genera la figura figuras/T1_softmax_temperatura.png, que ilustra ese afilado/aplanamiento de la distribución.

In [1]:
# T1: código aprobado por el crítico
# T1: Softmax con temperatura (NumPy, estable numéricamente) + entropía en bits
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Datos del enunciado
# ------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]

# ------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (max-shift)
# p_i = exp(z_i / T) / sum_j exp(z_j / T)
# ------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura T debe ser > 0.")
    escalados = z / T
    m = np.max(escalados)              # resta del máximo -> estabilidad
    exps = np.exp(escalados - m)
    return exps / np.sum(exps)

def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]                       # términos 0*log2(0) = 0
    return float(-np.sum(p * np.log2(p)))

# ------------------------------------------------------------------
# Cálculo de distribuciones y entropías
# ------------------------------------------------------------------
distribuciones = {}
entropias = {}
for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    distribuciones[f"T={T}"] = p.tolist()
    entropias[f"T={T}"] = entropia_bits(p)

sumas = {k: float(np.sum(v)) for k, v in distribuciones.items()}

# Verificación: T=1 debe coincidir con la softmax estándar de los logits
exps_std = np.exp(logits - np.max(logits))
softmax_estandar = exps_std / np.sum(exps_std)
diff_T1 = float(np.max(np.abs(softmax_estandar - np.array(distribuciones["T=1.0"]))))

# ------------------------------------------------------------------
# Tabla con cuatro decimales
# ------------------------------------------------------------------
tabla = {}
for T in temperaturas:
    key = f"T={T}"
    tabla[key] = {
        "distribucion_4dec": [round(v, 4) for v in distribuciones[key]],
        "entropia_bits_4dec": round(entropias[key], 4),
    }

df = pd.DataFrame(
    {
        "T": [f"T={T}" for T in temperaturas],
        **{t: [round(distribuciones[f"T={T}"][i], 4) for T in temperaturas]
           for i, t in enumerate(tokens)},
        "entropia_bits": [round(entropias[f"T={T}"], 4) for T in temperaturas],
        "suma": [round(sumas[f"T={T}"], 6) for T in temperaturas],
    }
)

# ------------------------------------------------------------------
# Contrato: resultados.json
# ------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": distribuciones,
    "entropias_bits": entropias,
    "tabla_4_decimales": tabla,
    "sumas_distribuciones": sumas,
    "verificacion_T1_vs_softmax_estandar_max_abs_diff": diff_T1,
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------------
# Figura: distribuciones según temperatura
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tokens))
ancho = 0.25
for i, T in enumerate(temperaturas):
    ax.bar(x + (i - 1) * ancho, distribuciones[f"T={T}"],
           width=ancho, label=f"T={T}")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura (logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0])")
ax.legend()
fig.tight_layout()
plt.savefig("softmax_temperatura.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print("=== T1: Softmax con temperatura ===")
print(f"Logits: {logits.tolist()}")
print()
print(df.to_string(index=False))
print()
for T in temperaturas:
    k = f"T={T}"
    print(f"{k}: suma = {sumas[k]:.10f} | H = {entropias[k]:.4f} bits")
print()
print(f"Verificación T=1 vs softmax estándar (max abs diff): {diff_T1:.3e}")
print("Archivos generados: resultados.json, softmax_temperatura.png")


=== T1: Softmax con temperatura ===
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

    T     t0     t1     t2     t3     t4     t5  entropia_bits  suma
T=0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000         0.9147   1.0
T=1.0 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037         1.7998   1.0
T=2.0 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294         2.2888   1.0

T=0.5: suma = 1.0000000000 | H = 0.9147 bits
T=1.0: suma = 1.0000000000 | H = 1.7998 bits
T=2.0: suma = 1.0000000000 | H = 2.2888 bits

Verificación T=1 vs softmax estándar (max abs diff): 0.000e+00
Archivos generados: resultados.json, softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se ordenan los tokens por probabilidad descendente — el orden natural ya es descendente: t0, t1, t2, t3, t4, t5 — y se acumula hasta alcanzar p = 0.9:

| Token | p original (T = 1) | Acumulada | ¿En el núcleo? | p renormalizada |
|---|---:|---:|:---:|---:|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | — |
| t5 | 0.0037 | 1.0000 | no | — |

Con t0–t2 la probabilidad acumulada llega a 0.8776, aún por debajo de 0.9; al incluir t3 alcanza 0.9688 ≥ 0.9. El conjunto mínimo que alcanza el umbral es por tanto {t0, t1, t2, t3}: **sobreviven t0, t1, t2 y t3**, y se descartan t4 y t5, cuya masa de probabilidad conjunta es 0.0312. Tras renormalizar (dividir entre 0.9688), la distribución del núcleo es t0 0.5694, t1 0.2095, t2 0.1270, t3 0.0941, con suma 1.0. Nótese que la acumulada del núcleo puede exceder p (aquí 0.9688 > 0.9), porque el criterio es el conjunto *mínimo* que alcanza el umbral, no quedarse por debajo de él. La celda genera la figura figuras/T2_top_p_nucleus.png con el núcleo resultante.

In [2]:
# T2: código aprobado por el crítico
# T2 — Muestreo de núcleo (top-p, p=0.9) sobre la distribución de T=1 de la Parte 1 (T1)
# Solo librerías permitidas. Sin red, sin subprocess, sin rutas absolutas.

import json
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. Cargar la distribución de T=1 calculada en T1 (entrada/T1/)
# ------------------------------------------------------------------
ruta_t1 = os.path.join("entrada", "T1", "resultados.json")
if os.path.exists(ruta_t1):
    with open(ruta_t1, "r", encoding="utf-8") as f:
        res_t1 = json.load(f)
    tokens = list(res_t1["tokens"])
    logits = np.array(res_t1["logits"], dtype=float)
    p_T1 = np.array(res_t1["distribuciones"]["T=1.0"], dtype=float)
    fuente = "entrada/T1/resultados.json"
else:
    # Respaldo: recalcular la softmax con T=1 (numéricamente estable)
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    p_T1 = np.exp(z) / np.exp(z).sum()
    fuente = "recalculada (respaldo)"

# Verificación: recomputar softmax(T=1) y comparar con lo cargado
z = logits / 1.0
z = z - z.max()
p_recalc = np.exp(z) / np.exp(z).sum()
diff_max = float(np.max(np.abs(p_T1 - p_recalc)))

# ------------------------------------------------------------------
# 2. Top-p (nucleus) con p = 0.9
#    Ordenar por probabilidad descendente, conservar el conjunto MÍNIMO
#    cuya probabilidad acumulada alcanza p (incluido el token que cruza
#    el umbral) y renormalizar sobre los supervivientes.
# ------------------------------------------------------------------
p_corte = 0.9

orden = np.argsort(-p_T1, kind="stable")          # índices en orden descendente
probs_ord = p_T1[orden]
tokens_ord = [tokens[i] for i in orden]
acumulada = np.cumsum(probs_ord)

# Número mínimo k de tokens con acumulada >= p (searchsorted con side='left'
# incluye el token en el que la acumulada iguala o supera el umbral)
k = int(np.searchsorted(acumulada, p_corte, side="left")) + 1
k = min(k, len(tokens))

idx_sup = orden[:k]
tokens_sup = [tokens[i] for i in idx_sup]
probs_sup = p_T1[idx_sup]
acum_sup = float(acumulada[k - 1])                # acumulada del núcleo (>= 0.9)

# Renormalización sobre los supervivientes
probs_renorm = probs_sup / probs_sup.sum()
suma_renorm = float(probs_renorm.sum())

tokens_desc = [tokens[i] for i in orden[k:]]
masa_desc = float(p_T1[orden[k:]].sum())

# Acumulada por token superviviente (para la tabla)
acum_por_sup = [float(acumulada[j]) for j in range(k)]

# ------------------------------------------------------------------
# 3. Escribir resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "p": p_corte,
    "temperatura": 1.0,
    "fuente_distribucion": fuente,
    "tokens": tokens,
    "logits": logits.tolist(),
    "distribucion_T1": p_T1.tolist(),
    "orden_descendente": tokens_ord,
    "probs_ordenadas_descendente": probs_ord.tolist(),
    "prob_acumulada_ordenada": acumulada.tolist(),
    "tokens_supervivientes_top_p": tokens_sup,
    "probs_supervivientes_originales": probs_sup.tolist(),
    "prob_acumulada_por_superviviente": acum_por_sup,
    "prob_acumulada_conjunto_superviviente": acum_sup,
    "cumple_acumulada_ge_p": bool(acum_sup >= p_corte),
    "distribucion_renormalizada": probs_renorm.tolist(),
    "suma_distribucion_renormalizada": suma_renorm,
    "tokens_descartados": tokens_desc,
    "masa_probabilidad_descartada": masa_desc,
    "tabla_resumen": [
        {
            "token": tokens_sup[j],
            "prob_original_T1": float(probs_sup[j]),
            "prob_acumulada": acum_por_sup[j],
            "prob_renormalizada": float(probs_renorm[j]),
        }
        for j in range(k)
    ],
    "verificacion_softmax_T1_max_abs_diff": diff_max,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------------
# 4. Figura: distribución original vs. núcleo top-p renormalizado
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

x_full = np.arange(len(tokens))
colores = ["darkorange" if i in set(idx_sup) else "lightgray" for i in range(len(tokens))]
axes[0].bar(x_full, p_T1, color=colores, edgecolor="black", linewidth=0.5)
axes[0].set_xticks(x_full)
axes[0].set_xticklabels(tokens)
axes[0].set_ylim(0, p_T1.max() * 1.18)
axes[0].set_ylabel("probabilidad")
axes[0].set_title("Distribución original (T=1)\n(naranja = supervivientes top-p)")
for i, v in enumerate(p_T1):
    axes[0].text(i, v + 0.008, f"{v:.4f}", ha="center", fontsize=8)

x_sup = np.arange(k)
axes[1].bar(x_sup, probs_renorm, color="darkorange", edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x_sup)
axes[1].set_xticklabels(tokens_sup)
axes[1].set_ylim(0, probs_renorm.max() * 1.18)
axes[1].set_ylabel("probabilidad renormalizada")
axes[1].set_title(f"Núcleo top-p (p={p_corte}): {k} tokens\nacumulada = {acum_sup:.4f} >= {p_corte}")
for i, v in enumerate(probs_renorm):
    axes[1].text(i, v + 0.008, f"{v:.4f}", ha="center", fontsize=8)

fig.suptitle("Muestreo de núcleo (top-p) sobre la distribución de T=1", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("top_p_nucleus.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 5. Resumen breve
# ------------------------------------------------------------------
print("=== T2: Muestreo de núcleo top-p (p=0.9) sobre la distribución T=1 ===")
print(f"Fuente de la distribución: {fuente}")
print("Orden descendente y acumulada:")
for t, v, a in zip(tokens_ord, probs_ord, acumulada):
    print(f"  {t}: p={v:.6f}  acum={a:.6f}")
print(f"Tokens supervivientes (conjunto mínimo con acumulada >= {p_corte}): {tokens_sup}")
print(f"Probabilidad acumulada del núcleo: {acum_sup:.6f} (>= 0.9: {acum_sup >= p_corte})")
print("Distribución renormalizada:")
for t, pr in zip(tokens_sup, probs_renorm):
    print(f"  {t}: {pr:.6f}")
print(f"Suma de la distribución renormalizada: {suma_renorm:.6f}")
print(f"Tokens descartados: {tokens_desc} (masa descartada = {masa_desc:.6f})")
print(f"Verificación softmax T=1 (máx. dif. absoluta): {diff_max:.2e}")
print("Figura guardada: top_p_nucleus.png | Resultados en resultados.json")


=== T2: Muestreo de núcleo top-p (p=0.9) sobre la distribución T=1 ===
Fuente de la distribución: entrada/T1/resultados.json
Orden descendente y acumulada:
  t0: p=0.551623  acum=0.551623
  t1: p=0.202931  acum=0.754553
  t2: p=0.123084  acum=0.877637
  t3: p=0.091183  acum=0.968820
  t4: p=0.027464  acum=0.996283
  t5: p=0.003717  acum=1.000000
Tokens supervivientes (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Probabilidad acumulada del núcleo: 0.968820 (>= 0.9: True)
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.000000
Tokens descartados: ['t4', 't5'] (masa descartada = 0.031180)
Verificación softmax T=1 (máx. dif. absoluta): 0.00e+00
Figura guardada: top_p_nucleus.png | Resultados en resultados.json


## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). La celda genera el gráfico de barras (figuras/T3_T3_frecuencias_observadas_vs_teoricas.png) que compara las frecuencias observadas con las probabilidades teóricas:

| Token | Prob. teórica | Conteo observado | Frecuencia observada | Obs − teo |
|---|---:|---:|---:|---:|
| t0 | 0.5694 | 5682 | 0.5682 | −0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | −0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | +0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | −0.0012 |

Las frecuencias observadas suman 1.0 y siguen de cerca a las teóricas: la mayor desviación absoluta es 0.0026 (en t2) y la distancia de variación total es 0.0026. La divergencia KL de la distribución empírica respecto de la teórica es **5.0069e-05 bits** (3.4705e-05 nats), un valor prácticamente nulo. Esto confirma que el muestreo implementa fielmente la distribución objetivo y que las diferencias observadas corresponden al ruido estadístico esperable con 10 000 muestras; en el gráfico, las barras de cada par teoría–observación son casi indistinguibles.

In [3]:
# T3: código aprobado por el crítico
# T3: Comprobación empírica del muestreo top-p (Parte 3)
# - Toma 10,000 muestras de la distribución renormalizada de T2 con numpy.random.default_rng(0)
# - Grafica frecuencias observadas vs probabilidades teóricas (barras)
# - Calcula la divergencia KL en bits de la empírica respecto de la teórica

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

N_MUESTRAS = 10000
SEMILLA = 0
P_TOP = 0.9
T_TEMP = 1.0

# ------------------------------------------------------------------
# 1) Cargar la distribución renormalizada de T2 (con respaldo)
# ------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    tokens_sup = list(t2["tokens_supervivientes_top_p"])
    p_teorica = np.array(t2["distribucion_renormalizada"], dtype=float)
    fuente = "entrada/T2/resultados.json"
else:
    # Respaldo: recalcular softmax(T=1) + top-p(0.9) + renormalización
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    p1 = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p1)
    acum = np.cumsum(p1[orden])
    k = int(np.searchsorted(acum, P_TOP) + 1)
    idx = np.sort(orden[:k])
    p_teorica = p1[idx] / p1[idx].sum()
    tokens_sup = [f"t{i}" for i in idx]
    fuente = "recalculada (softmax T=1 + top-p 0.9)"

assert abs(p_teorica.sum() - 1.0) < 1e-12, "La distribución teórica debe sumar 1"

# ------------------------------------------------------------------
# 2) Muestreo: 10,000 muestras con default_rng(0)
# ------------------------------------------------------------------
rng = np.random.default_rng(SEMILLA)
muestras = rng.choice(len(tokens_sup), size=N_MUESTRAS, p=p_teorica)

conteos = np.bincount(muestras, minlength=len(tokens_sup))
frec_obs = conteos / N_MUESTRAS

# ------------------------------------------------------------------
# 3) Divergencia KL en bits: D_KL(empírica || teórica)
#    (términos con frecuencia observada 0 aportan 0)
# ------------------------------------------------------------------
mask = frec_obs > 0
kl_bits = float(np.sum(frec_obs[mask] * np.log2(frec_obs[mask] / p_teorica[mask])))
kl_nats = float(np.sum(frec_obs[mask] * np.log(frec_obs[mask] / p_teorica[mask])))
tv = float(0.5 * np.sum(np.abs(frec_obs - p_teorica)))  # distancia total (informativa)

# ------------------------------------------------------------------
# 4) Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# ------------------------------------------------------------------
x = np.arange(len(tokens_sup))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - ancho / 2, frec_obs, ancho, color="#4C72B0",
            label="Frecuencia observada (10,000 muestras)")
b2 = ax.bar(x + ancho / 2, p_teorica, ancho, color="#DD8452",
            label="Probabilidad teórica (top-p renormalizada)")
for rect, val in list(zip(b1, frec_obs)) + list(zip(b2, p_teorica)):
    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 0.006,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens_sup)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_title(f"Muestreo top-p (T=1, p={P_TOP}): {N_MUESTRAS} muestras — "
             f"KL(empírica||teórica) = {kl_bits:.6f} bits")
ax.set_ylim(0, max(frec_obs.max(), p_teorica.max()) * 1.18)
ax.legend()
fig.tight_layout()
fig.savefig("T3_frecuencias_observadas_vs_teoricas.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 5) Escribir resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T3_comprobacion_empirica",
    "n_muestras": N_MUESTRAS,
    "semilla": SEMILLA,
    "generador": "numpy.random.default_rng(0)",
    "fuente_distribucion": fuente,
    "tokens_supervivientes_top_p": tokens_sup,
    "probabilidades_teoricas_renormalizadas": p_teorica.tolist(),
    "conteos_observados": conteos.tolist(),
    "frecuencias_observadas": frec_obs.tolist(),
    "suma_frecuencias_observadas": float(frec_obs.sum()),
    "divergencia_KL_bits_empirica_respecto_teorica": kl_bits,
    "divergencia_KL_nats_empirica_respecto_teorica": kl_nats,
    "distancia_variacion_total": tv,
    "tabla_comparativa": [
        {
            "token": tok,
            "prob_teorica": float(pt),
            "conteo_observado": int(co),
            "frecuencia_observada": float(fo),
            "diferencia_obs_menos_teo": float(fo - pt),
        }
        for tok, pt, co, fo in zip(tokens_sup, p_teorica, conteos, frec_obs)
    ],
    "figura": "T3_frecuencias_observadas_vs_teoricas.png",
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------------
# 6) Resumen
# ------------------------------------------------------------------
print("=== T3: Comprobación empírica del muestreo top-p ===")
print(f"Muestras: {N_MUESTRAS} | Generador: numpy.random.default_rng({SEMILLA})")
print(f"Fuente de la distribución: {fuente}")
print(f"Tokens supervivientes: {tokens_sup}")
print("Token |  teórica  |  observada | conteo")
for tok, pt, fo, co in zip(tokens_sup, p_teorica, frec_obs, conteos):
    print(f"  {tok}  | {pt:.6f} | {fo:.6f} | {co}")
print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits "
      f"({kl_nats:.6f} nats) -> cercana a 0: {kl_bits < 0.01}")
print(f"Distancia de variación total = {tv:.6f}")
print("Figura guardada: T3_frecuencias_observadas_vs_teoricas.png")
print("Resultados escritos en resultados.json")


=== T3: Comprobación empírica del muestreo top-p ===
Muestras: 10000 | Generador: numpy.random.default_rng(0)
Fuente de la distribución: entrada/T2/resultados.json
Tokens supervivientes: ['t0', 't1', 't2', 't3']
Token |  teórica  |  observada | conteo
  t0  | 0.569376 | 0.568200 | 5682
  t1  | 0.209462 | 0.209300 | 2093
  t2  | 0.127045 | 0.129600 | 1296
  t3  | 0.094117 | 0.092900 | 929
Divergencia KL (empírica || teórica) = 0.000050 bits (0.000035 nats) -> cercana a 0: True
Distancia de variación total = 0.002555
Figura guardada: T3_frecuencias_observadas_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

La temperatura reescala los logits antes de la softmax ($p_i \propto \exp(z_i/T)$) y por tanto controla cuánto pesan las diferencias entre logits.

**Cuando T → 0.** Dividir por una T muy pequeña amplifica las diferencias $z_i/T$: el término del logit máximo (t0, z = 2.0) domina la suma y la probabilidad se concentra en él, hasta degenerar en una delta: p(t0) → 1 y el resto de los tokens → 0. Los resultados de la Parte 1 muestran esa trayectoria: p(t0) crece de 0.3583 (T = 2) a 0.5516 (T = 1) y 0.8231 (T = 0.5), mientras la entropía cae de 2.2888 a 1.7998 y 0.9147 bits. En el límite matemático T → 0 la entropía tiende a 0 bits, el valor de una distribución determinista (una delta).

**Cuando T → ∞.** Entonces $z_i/T \to 0$ para todo token, todos los exponenciales tienden al mismo valor y la distribución converge a la uniforme sobre el vocabulario, con la misma probabilidad para cada uno de los seis tokens. La evidencia medida apunta en esa dirección: al pasar de T = 1 a T = 2 la probabilidad máxima baja de 0.5516 a 0.3583, la mínima sube de 0.0037 a 0.0294, y la entropía sube a 2.2888 bits, acercándose al máximo posible para seis tokens; la tendencia monótona de la entropía (0.9147 → 1.7998 → 2.2888 bits al aumentar T) es la que continúa hacia ese límite uniforme.

**Por qué T → 0 equivale a la decodificación voraz.** La decodificación voraz (greedy) elige en cada paso el token de mayor logit (argmax). Cuando T → 0, la softmax pone toda la masa precisamente en ese token, de modo que muestrear de la distribución devuelve el token voraz con probabilidad 1: el muestreo se vuelve determinista e idéntico a greedy. La entropía es la evidencia cuantitativa de esta pérdida de aleatoriedad: mide la incertidumbre de la elección y, al reducir T, desciende monótonamente (2.2888 → 1.7998 → 0.9147 bits) hacia el 0 de una delta; ya con T = 0.5 el muestreo elige t0 con probabilidad 0.8231. En ese régimen el núcleo top-p se reduce al token líder y, en la implementación, resulta esencial restar el máximo de los logits, pues exp(z_i/T) desborda para logits positivos cuando T es muy pequeña; la verificación de la Parte 1 (diferencia máxima 0.0 frente a la softmax directa) confirma que este ajuste no cambia el resultado.